COSC 301 Project
Group 4
Instacart Market Basket Analysis

**Importing Files**

The way to run this program is by running bash scripts/get_data.sh in terminal 

In [1]:
import pandas as pd               # for data manipulation
import matplotlib.pyplot as plt   # for plotting 
import seaborn as sns             # for statistical graph
import sqlite3                    # for connecting to the database

In [2]:
orders = pd.read_csv('../raw_data/orders.csv' )
products = pd.read_csv('../raw_data/products.csv')
order_products_prior = pd.read_csv('../raw_data/order_products__prior.csv')
aisles = pd.read_csv('../raw_data/aisles.csv')
order_products_train = pd.read_csv('../raw_data/order_products__train.csv')

In [4]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


# Aadil's code

In [5]:
prior_orders = pd.merge(orders, order_products_prior, on='order_id', how='inner')
print(prior_orders.memory_usage(deep=True).sum() / (1024**3))
print(len(orders))
print(len(products))
print(len(order_products_prior))
print(len(aisles))
print(f"prior_orders: {len(prior_orders)} rows")

3.8060785699635744
3421083
49688
32434489
134
prior_orders: 32434489 rows


## Cleaning data

How to clean drop columns remove na values or rows or columns maybe check how many na values in a row or column and only remove if greater than 5 or 10% or maybe remove all rows that have an na in them

In [6]:
#code for removing na 
prior_orders_cleaned = prior_orders.dropna()

print(f"prior_orders_cleaned: {len(prior_orders_cleaned)} rows")

prior_orders_cleaned: 30356421 rows


## Research Question 1

How accurately will we recommend the top 5 products a user is most likely to purchase in their next order based on their previous orders (using the order_product_prior csv file)?

Steps to answer the question

Step 1: create dataframe with all except last order of each user. use a loop to create dataframe for each user.

Step 2: Calculate top 5 products based on frequency of ordering the product that the user has order in all of their order excluding their latest order OR Use ML algorithm to recommend 5 products based on what user has previously ordered OR BOTH

Step 3: recommend top 5 products a user will order in the next order based on previous orders. compute accuracy by comparing the recommended 5 products with the actual orders ordered in the last order of the user

In [7]:
#step 1

#making the connection to a created database
connection = sqlite3.connect("instacart.db")

#sending dataframe to the sql database
prior_orders.to_sql("prior_orders", connection, if_exists = "replace", index = False)

query = """
        SELECT order_id, user_id, product_id, order_number
        FROM (
            SELECT 
                order_id,
                user_id,
                product_id,
                order_number,
                MAX(order_number) OVER (PARTITION BY user_id) AS max_order_number
            FROM prior_orders 
        )
        WHERE order_number < max_order_number 
    """
    
df_without_last_orders_of_users = pd.read_sql(query, connection)

df_without_last_orders_of_users.head()

,order_id,user_id,product_id,order_number
0,2539329,1,196,1
1,2539329,1,14084,1
2,2539329,1,12427,1
3,2539329,1,26088,1
4,2539329,1,26405,1


In [8]:
product_counts = (
                df_without_last_orders_of_users
                .groupby(["user_id", "product_id"])
                .size()
                .reset_index(name = "order_count")
)

product_counts["rank"] = (
                        product_counts
                        .groupby("user_id")["order_count"]
                        .rank(method = "first", ascending = False)
)

top5_products = product_counts[product_counts['rank'] <= 5]

top5_products.head(10)

,user_id,product_id,order_count,rank
0,1,196,9,1.0
1,1,10258,8,3.0
3,1,12427,9,2.0
4,1,13032,2,5.0
8,1,25133,7,4.0
17,2,1559,5,4.0
35,2,12000,5,5.0
66,2,24852,6,3.0
78,2,32792,9,1.0
101,2,47209,7,2.0


# Continuation of Raunak's code

In [9]:
# Join prior order-products with orders to get user_id + order_number
op_prior = order_products_prior.merge(
    orders[['order_id', 'user_id', 'order_number']],
    on='order_id',
    how='left'
)

# add product meta (name/aisle/department)
prod_meta = products[['product_id', 'product_name', 'aisle_id', 'department_id']].copy()
op_prior = op_prior.merge(prod_meta, on='product_id', how='left')

# now we have a dataframe with info order, products in the order, and user who made the order.
op_prior.head()


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,product_name,aisle_id,department_id
0,2,33120,1,1,202279,3,Organic Egg Whites,86,16
1,2,28985,2,1,202279,3,Michigan Organic Kale,83,4
2,2,9327,3,0,202279,3,Garlic Powder,104,13
3,2,45918,4,1,202279,3,Coconut Butter,19,13
4,2,30035,5,0,202279,3,Natural Sweetener,17,13


In [10]:
# Per-user, per-product stats from PRIOR history
# calculating the numbers of times a user bought a product, the last order number they bought it in, 
# and the reorder rate for that product for that user.
user_prod_stats = (
    op_prior.groupby(['user_id', 'product_id'], as_index=False)
    .agg(
        times_bought=('order_id', 'count'),
        last_order_number=('order_number', 'max'),
        reorder_rate=('reordered', 'mean')
    )
)

# merge names for readability
user_prod_stats = user_prod_stats.merge(prod_meta, on='product_id', how='left')

# recommendation function based on frequency of a product being ordered; accepts user id and number of
# recommendations to make
def recommend_top5_frequency(user_id: int, n: int = 5):
    """
    Non-ML baseline:
    - prefer items bought often
    - prefer items bought recently
    - prefer items with high reorder_rate
    """
    df = user_prod_stats[user_prod_stats['user_id'] == user_id].copy()
    if df.empty:
        return pd.DataFrame(columns=['product_id','product_name','times_bought','reorder_rate','last_order_number'])

    df = df.sort_values(
        by=['times_bought', 'last_order_number', 'reorder_rate'],
        ascending=[False, False, False]
    )
    return df[['user_id','product_id','product_name','times_bought','reorder_rate','last_order_number']].head(n)

# try
recommend_top5_frequency(1)

,user_id,product_id,product_name,times_bought,reorder_rate,last_order_number
0,1,196,Soda,10,0.900000,10
3,1,12427,Original Beef Jerky,10,0.900000,10
1,1,10258,Pistachios,9,0.888889,10
8,1,25133,Organic String Cheese,8,0.875000,10
4,1,13032,Cinnamon Toast Crunch,3,0.666667,10


In [11]:
# Global popularity + reorder rate per product (from PRIOR)
global_prod_stats = (
    op_prior.groupby('product_id', as_index=False)
    .agg(
        global_times=('order_id', 'count'),
        global_reorder_rate=('reordered', 'mean')
    )
).merge(prod_meta, on='product_id', how='left')

# User aisle preferences
#aisle_count is the number of a times the aisle was mentioned in a prior order for a user.
user_aisle_pref = (
    op_prior.groupby(['user_id', 'aisle_id'], as_index=False)
    .agg(aisle_count=('order_id', 'count'))
)

# Popular products within each aisle
#this is different from user_aisle_pref because it is number of times a product in an aisle was ordered in general and not specific to a user.
aisle_prod_pop = (
    op_prior.groupby(['aisle_id', 'product_id'], as_index=False)
    .agg(aisle_prod_count=('order_id', 'count'))
).merge(global_prod_stats, on=['product_id','aisle_id'], how='left')


def recommend_top5_aisle(user_id: int, n: int = 5, top_aisles: int = 3):
    """
    Content-based recommender:
    1) find user's top aisles
    2) recommend popular products from those aisles
       (rank by aisle popularity + global reorder rate)
    """
    # products already seen by user
    seen = set(user_prod_stats[user_prod_stats['user_id'] == user_id]['product_id'].unique())

    # top aisles for user
    topA = (
        user_aisle_pref[user_aisle_pref['user_id'] == user_id]
        .sort_values('aisle_count', ascending=False)
        .head(top_aisles)['aisle_id']
        .tolist()
    )
        
    if not topA:
        return pd.DataFrame(columns=['product_id','product_name','aisle_id','aisle_prod_count','global_reorder_rate'])

    popular_prod_AND_in_top_aisle_for_user = aisle_prod_pop[aisle_prod_pop['aisle_id'].isin(topA)].copy()
    
    # recommend either: (A) reorders allowed, or (B) new items only
    # Here: NEW items only (more interesting). Comment this out if you want reorders too.
    #popular_prod_AND_in_top_aisle_for_user = popular_prod_AND_in_top_aisle_for_user[~popular_prod_AND_in_top_aisle_for_user['product_id'].isin(seen)]

    #sorting products to recommend by aisle
    popular_prod_AND_in_top_aisle_for_user = popular_prod_AND_in_top_aisle_for_user.sort_values(
        by=['aisle_prod_count', 'global_reorder_rate', 'global_times'],
        ascending=[False, False, False]
    )

    return popular_prod_AND_in_top_aisle_for_user[['product_id','product_name','aisle_id','aisle_prod_count','global_reorder_rate']].head(n)

# try
recommend_top5_aisle(1)

,product_id,product_name,aisle_id,aisle_prod_count,global_reorder_rate
26966,196,Soda,77,35791,0.776480
27077,10957,Fridge Pack Cola,77,18269,0.715255
8257,3599,Baked Aged White Cheddar Rice and Corn Puffs,23,13691,0.643635
8337,15902,100 Calorie Per Bag Popcorn,23,12822,0.678209
44079,35140,Organic Whole Cashews,117,12816,0.605025


In [12]:
# Build user -> set of products in their TRAIN order (the "next order" we are trying to predict)
op_train = order_products_train.merge(
    orders[['order_id', 'user_id']],
    on='order_id',
    how='left'
)

truth = (
    op_train.groupby('user_id')['product_id']
    .apply(lambda s: set(s.values))
    .to_dict()
)

import random

def precision_at_k(recs, truth_set, k=5):
    recs = list(recs)[:k]
    if not recs:
        return 0.0
    hits = len(set(recs) & truth_set)
    return hits / k

def recall_at_k(recs, truth_set, k=5):
    recs = list(recs)[:k]
    if not truth_set:
        return 0.0
    hits = len(set(recs) & truth_set)
    return hits / len(truth_set)

def eval_recommender(user_ids, recommender_fn, k=5):
    ps, rs = [], []
    for uid in user_ids:
        if uid not in truth:
            continue
        rec_df = recommender_fn(uid, n=k)
        recs = rec_df['product_id'].tolist() if 'product_id' in rec_df else []
        ps.append(precision_at_k(recs, truth[uid], k))
        rs.append(recall_at_k(recs, truth[uid], k))
    return (sum(ps)/len(ps), sum(rs)/len(rs)) if ps else (0.0, 0.0)

# sample users for speed
all_users = list(truth.keys())
sample_users = random.sample(all_users, k=min(2000, len(all_users)))

p1, r1 = eval_recommender(sample_users, recommend_top5_frequency, k=5)
p2, r2 = eval_recommender(sample_users, recommend_top5_aisle, k=5)

print("Baseline Frequency  P@5:", round(p1,4), " R@5:", round(r1,4))
print("Aisle Content-Based P@5:", round(p2,4), " R@5:", round(r2,4))

Baseline Frequency  P@5: 0.3647  R@5: 0.23
Aisle Content-Based P@5: 0.0926  R@5: 0.0493


Based on the above metrics printed, the performance is better for the frequency based method rather than the aisle popularity method of recommendation.

## Research Question 2

**How does recommendation performance change with user history size?**

This question explores whether users with more order history receive better recommendations. Intuitively, a frequency-based recommender should perform better for users with many prior orders because it has more data to learn their preferences. Users with little history (or cold start users) may get worse recommendations since the model has less signal to work with.

**User groupings:**
- **Few prior orders (< 5)**: Users with fewer than 5 prior orders—includes near cold-start users
- **Moderate history (5 to 20 prior orders)**: Users with enough history to establish some patterns
- **Heavy users (> 20 prior orders)**: Users with extensive order history

**Approach:** We use the same frequency-based recommender from Research Question 1 and evaluate precision@5, recall@5, and hit rate@5 separately for each history group.

In [13]:
# Compute each user's history size: number of prior orders (before the held-out train order)
# op_prior has all prior order-product pairs; we count distinct order_ids per user
user_prior_order_count = op_prior.groupby('user_id')['order_id'].nunique().reset_index()
user_prior_order_count.columns = ['user_id', 'prior_order_count']

# Assign users to history groups
def assign_history_group(count):
    if count < 5:
        return '< 5 prior orders'
    elif count <= 20:
        return '5 to 20 prior orders'
    else:
        return '> 20 prior orders'

user_prior_order_count['history_group'] = user_prior_order_count['prior_order_count'].apply(assign_history_group)

# Keep only users who have a train order (ground truth) for evaluation
user_history = user_prior_order_count[user_prior_order_count['user_id'].isin(truth.keys())].copy()

# Cold start: users in truth (train) but not in prior orders - excluded from RQ2
cold_start = set(truth.keys()) - set(user_prior_order_count['user_id'])
if cold_start:
    print(f"Cold start users (in train but no prior orders): {len(cold_start)} — excluded from RQ2")
print()
print("User distribution by history group:")
print(user_history['history_group'].value_counts().sort_index())
print()
print("Prior order count stats:")
print(user_history['prior_order_count'].describe())


User distribution by history group:
history_group
5 to 20 prior orders    72975
< 5 prior orders        27739
> 20 prior orders       30495
Name: count, dtype: int64

Prior order count stats:
count    131209.000000
mean         15.603937
std          16.661077
min           3.000000
25%           5.000000
50%           9.000000
75%          19.000000
max          99.000000
Name: prior_order_count, dtype: float64


### Performance by history group

We evaluate the same frequency-based recommender on each group. Metrics:
- **Precision@5**: Of the 5 recommendations, what fraction are in the user's actual next order?
- **Recall@5**: Of the products in the user's actual order, what fraction did we recommend?
- **Hit rate@5**: What fraction of users got at least one correct recommendation?

In [17]:
# Evaluate recommender per history group using same method as Research Question 1
# Add hit rate at k: fraction of users with at least one correct recommendation
import time

def hit_rate_at_k(recs, truth_set, k=5):
    recs = list(recs)[:k]
    if not recs:
        return 0.0
    hits = len(set(recs) & truth_set)
    return 1.0 if hits > 0 else 0.0

def eval_recommender_rq2(user_ids, recommender_fn, k=5):
    """Returns (avg_precision, avg_recall, avg_hit_rate) for the given user list."""
    ps, rs, hrs = [], [], []
    start_time = time.time()
    total_users = len(user_ids)
    for i, uid in enumerate(user_ids, 1):
        if uid not in truth:
            continue
        rec_df = recommender_fn(uid, n=k)
        recs = rec_df['product_id'].tolist() if 'product_id' in rec_df and len(rec_df) > 0 else []
        ps.append(precision_at_k(recs, truth[uid], k))
        rs.append(recall_at_k(recs, truth[uid], k))
        hrs.append(hit_rate_at_k(recs, truth[uid], k))
        if i % 50 == 0:
            elapsed = time.time() - start_time
            avg_time_per_user = elapsed / i
            remaining = total_users - i
            eta = avg_time_per_user * remaining
            print(f"[Progress] {i}/{total_users} users processed "
                  f"({(i/total_users)*100:.2f}%) | "
                  f"Elapsed: {elapsed:.2f}s | "
                  f"ETA: {eta:.2f}s")
    total_time = time.time() - start_time
    print(f"[Done] Processed {total_users} users in {total_time:.2f} seconds")
    n = len(ps)
    if n == 0:
        return 0.0, 0.0, 0.0
    return sum(ps)/n, sum(rs)/n, sum(hrs)/n

# Evaluate each history group with recommend_top5_frequency (same as RQ1)
results_rq2 = []
group_order = ['< 5 prior orders', '5 to 20 prior orders', '> 20 prior orders']
for grp in group_order:
    users_in_group = user_history[user_history['history_group'] == grp]['user_id'].tolist()
    if not users_in_group:
        continue
    print(f"\nEvaluating group: {grp} | Users: {len(users_in_group)}")
    p5, r5, hr5 = eval_recommender_rq2(users_in_group, recommend_top5_frequency, k=5)
    results_rq2.append({
        'history_group': grp,
        'n_users': len(users_in_group),
        'precision_at_5': round(p5, 4),
        'recall_at_5': round(r5, 4),
        'hit_rate_at_5': round(hr5, 4)
    })

rq2_df = pd.DataFrame(results_rq2)
rq2_df


Evaluating group: < 5 prior orders | Users: 27739
[Progress] 50/27739 users processed (0.18%) | Elapsed: 0.50s | ETA: 277.24s
[Progress] 100/27739 users processed (0.36%) | Elapsed: 0.99s | ETA: 274.23s
[Progress] 150/27739 users processed (0.54%) | Elapsed: 1.47s | ETA: 269.64s
[Progress] 200/27739 users processed (0.72%) | Elapsed: 1.93s | ETA: 265.59s
[Progress] 250/27739 users processed (0.90%) | Elapsed: 2.41s | ETA: 265.33s
[Progress] 300/27739 users processed (1.08%) | Elapsed: 2.89s | ETA: 263.94s
[Progress] 350/27739 users processed (1.26%) | Elapsed: 3.35s | ETA: 262.54s
[Progress] 400/27739 users processed (1.44%) | Elapsed: 3.84s | ETA: 262.12s
[Progress] 450/27739 users processed (1.62%) | Elapsed: 4.31s | ETA: 261.21s
[Progress] 500/27739 users processed (1.80%) | Elapsed: 4.77s | ETA: 259.91s
[Progress] 550/27739 users processed (1.98%) | Elapsed: 5.24s | ETA: 259.14s
[Progress] 600/27739 users processed (2.16%) | Elapsed: 5.71s | ETA: 258.15s
[Progress] 650/27739 users

,history_group,n_users,precision_at_5,recall_at_5,hit_rate_at_5
0,< 5 prior orders,27739,0.3097,0.2259,0.7302
1,5 to 20 prior orders,72975,0.3571,0.2360,0.7874
2,> 20 prior orders,30495,0.3872,0.2244,0.8175


In [ ]:
# Bar chart: performance by history group
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(rq2_df))
width = 0.25
ax.bar([i - width for i in x], rq2_df['precision_at_5'], width, label='Precision@5')
ax.bar(x, rq2_df['recall_at_5'], width, label='Recall@5')
ax.bar([i + width for i in x], rq2_df['hit_rate_at_5'], width, label='Hit rate@5')
ax.set_xticks(x)
ax.set_xticklabels(rq2_df['history_group'], rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_title('Recommendation performance by user history size')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Interpretation

- **Users with more order history generally receive better recommendations.** Precision, recall, and hit rate tend to be higher for the "> 20 prior orders" group than for "< 5 prior orders." This matches intuition: the frequency-based recommender has more data to identify recurring purchases for heavy users.

- **Cold start / low-history users perform worse.** Users with fewer than 5 prior orders have less purchase history, so the model has weaker signals to predict their next order. Recommendations may rely on limited product frequency patterns that are noisier.

- **The pattern is consistent across metrics.** If hit rate, precision, and recall all improve with history size, it suggests the effect is robust—it's not just that heavy users get slightly more hits; they get more relevant recommendations overall.

- **Practical implication:** For production systems, consider different strategies for low-history users (e.g., popularity-based fallbacks or onboarding flows) while relying on personalized frequency-based recommendations for users with sufficient history.